[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/07_wrap_up_end_to_end.ipynb)


# Agentic Systems Foundations
## Notebook 07: End to End, and What to Take Away
**Duration:** 5 min &nbsp;|&nbsp; **Mode:** Q&A

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** all of it — one agent, one goal, every piece we built visible in the trace.

> **Requires `OPENAI_API_KEY`.** These notebooks call a real model —
> there is no simulated fallback, on purpose.


In [ ]:
# ============================================================
# BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Works in Colab, a local venv, or a bare Jupyter. It installs whatever is
# actually MISSING rather than assuming a particular environment — checking by
# import is the only reliable test.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork

# import name -> pip package name
REQUIRED = {
    "openai": "openai",
    "dotenv": "python-dotenv",
    "jsonschema": "jsonschema",
    "langchain_core": "langchain-core",
    "langchain_openai": "langchain-openai",
    "langgraph": "langgraph",
}

def _present(module: str) -> bool:
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError, ModuleNotFoundError):
        return False

missing = sorted({pkg for mod, pkg in REQUIRED.items() if not _present(mod)})
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
else:
    print("dependencies: all present")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# YOUR API KEY  (required — there is no offline fallback)
# ============================================================
# Every notebook in this session calls a REAL model. There is deliberately no
# simulated fallback: a fake model can show you the shape of an agent loop, but
# it cannot show you how a real one behaves when your tool descriptions are
# ambiguous or your schema is too loose — and that behaviour is the subject of
# the session.
#
#   Colab : sidebar -> key icon -> add a secret named OPENAI_API_KEY
#           -> toggle "Notebook access" ON -> re-run this cell
#   local : export OPENAI_API_KEY=sk-...   (or put it in a .env file)
import os

def _load_key() -> bool:
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

if not _load_key():
    raise RuntimeError(
        "OPENAI_API_KEY is not set — this notebook calls a real model.\n"
        "Colab: sidebar -> key icon -> add OPENAI_API_KEY -> Notebook access ON.\n"
        "Local: export OPENAI_API_KEY=sk-...  then restart the kernel."
    )

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
print()
print("These notebooks spend real tokens. Budgets are deliberately small.")

## The whole session in one cell

Everything from the last three hours, running at once. Watch the trace: state
carrying forward, tools acting, schemas validating, a skill scoping the choice,
control deciding when to stop.


In [ ]:
from agent_core import Agent

agent = Agent(verbose=True)   # verbose=True prints each step as the loop turns
result = agent.run(
    "Order ACME-1046 — I changed my mind and want a refund. "
    "Also, what would twelve months of that plan have cost?"
)

print()
print("=" * 72)
print("ANSWER:")
print(result.answer)

In [ ]:
# Every layer we built, visible in one object.
print("skill routed to :", result.skill)
print("tools called    :", " → ".join(result.tools_called()))
print("status          :", result.state.status.value)
print("stop reason     :", result.state.stop_reason)
print("steps           :", len(result.trace.steps))
print("error rate      :", f"{result.trace.error_rate():.0%}")
print("context growth  :", [s.context_tokens for s in result.trace.steps])
print()
print("EVIDENCE the answer must be built from:")
print(result.state.evidence())

## The four things worth keeping

**1. An agent is a loop over explicit state.**
Not a smarter model — the *same* model, called repeatedly, with the results of
its own actions appended to its input. Delete the write-back and it repeats
itself forever, perfectly rationally. That one line is the difference between an
LLM and an agent.

**2. Tools extend reach; structure makes it reliable.**
The schema is where you decide how much to trust the model. `enum` and `pattern`
are the highest-value lines you will write. Constrain at the boundary, coerce
what is unambiguous, and **write your errors for the model to read** — that is
what makes a retry loop converge instead of spin. And tools must never raise.

**3. Skills are how this survives growth.**
Tool choice degrades as the list grows, silently. Scope each job to the tools it
needs and route between them; that is how you reach twenty-five tools while no
single agent ever sees more than four. Route by default, compose only when one
request genuinely spans two jobs.

**4. Control and trace are not optional extras.**
`while True:` is a complete agent and an irresponsible one. Diagnostic
termination conditions before the budget backstop cost less and tell you more.
And because agent failures happen *across time* with nothing thrown, the trace is
the only debugging interface you have — an untraced failure is unreproducible and
therefore unfixable.


## Where this connects to last session

`search_docs` is a real retriever over the same Acme corpus you built a RAG
pipeline on. The entire C8 session — loading, chunking, retrieval, ranking —
collapsed into **one entry in a tool registry**.

That is the relationship, and it settles a question people waste a lot of time
on: **RAG is not an alternative to agents. It is a tool an agent calls.** The
better question is not "should I build RAG or an agent?" but "what does this
agent need to be able to look up?"

Notice too that the C8 failure mode came with it. An ungrounded agent answer is a
RAG hallucination one level up — same problem, now spread across several steps
and harder to see. The fix is the same: ground the answer in retrieved evidence,
and let it say "I don't know".


## If you take one habit away

**Read the trace before you change the prompt.**

The instinct when an agent misbehaves is to rewrite the system prompt. It is
usually wrong. Work the four questions in order:

```
1. Did it call the right tools?      No -> routing, descriptions, scoping
2. Did the calls succeed?            No -> schemas, arguments, tool bugs
3. Did it stop for a good reason?    No -> termination policy
4. Is the answer in the evidence?    No -> grounding, prompt
```

Only question 4 is a prompt problem. Three times out of four you will find the
bug somewhere above it.


In [ ]:
# Where to look next in this package. Each module opens with a WHY/WHAT/HOW
# docstring — they are written to be read, not skimmed.
import agent_core
print(agent_core.__doc__)

## Exercises and further reading

- `teaching/exercises.md` — graded practice per section, plus a capstone that
  adds a new tool, a new skill, and negative tests for both.
- `teaching/learner_handout.md` — cheat sheets for JSON Schema, termination
  conditions and the failure taxonomy, plus a glossary.
- `scripts/smoke_test.py` — the package's own verification, and a decent worked
  example of how to test an agent.

**Docs:**
[LangChain Agents](https://python.langchain.com/docs/concepts/agents/) ·
[LangChain Tools](https://python.langchain.com/docs/concepts/tools/) ·
[Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling) ·
[OpenAI function calling](https://platform.openai.com/docs/guides/function-calling) ·
[OpenAI Agents SDK](https://platform.openai.com/docs/guides/agents)

## Questions?
